<a href="https://colab.research.google.com/github/Arobnett/HDX-sources-and-more-API-connection/blob/main/notebooks/03d_assemble_feature_table.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03d Assemble Modeling Feature Table

This notebook reads the Step 03c cleaned country-month-year outputs and writes one wide modeling feature table plus validation reports. If the cleaned outputs are missing in a fresh Colab runtime, it runs the Step 03c cleaner first.


In [1]:
from pathlib import Path  # Work with filesystem paths.

REPO_URL = 'https://github.com/Arobnett/HDX-sources-and-more-API-connection.git'  # Store the GitHub repository URL.
REPO_DIR = Path('/content/HDX-sources-and-more-API-connection')  # Define the local Colab clone folder.

%cd /content
if not REPO_DIR.exists():  # Clone the repo when this runtime does not have it yet.
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/HDX-sources-and-more-API-connection
!git pull  # Refresh the Colab clone after GitHub edits.


/content
/content/HDX-sources-and-more-API-connection
Already up to date.


In [2]:
!git lfs install  # Enable Git LFS support in this Colab runtime.
!git lfs pull  # Download real large source files instead of pointer files.


Updated Git hooks.
Git LFS initialized.


In [3]:
import sys  # Access Python's module search path.
from pathlib import Path  # Represent repository paths independently of operating system.

PROJECT_ROOT = Path.cwd()  # Treat the cloned repository as the project root.
SRC_DIR = PROJECT_ROOT / 'src'  # Locate shared Python modules.
sys.path.insert(0, str(SRC_DIR))  # Make src modules importable in this Colab session.

from cleaning import clean_silver_directory  # Import the Step 03c cleaner for fresh runtimes.
from feature_assembly import assemble_feature_table  # Import the Step 03d feature assembly runner.
from paths import (
    CLEAN_DIR,
    FEATURE_REPORTS_DIR,
    MODEL_FEATURES_DIR,
    SILVER_DIR,
    SOURCE_CLEANING_RULES_PATH,
    ensure_output_directories,
)  # Import canonical paths.

print(f'Project root: {PROJECT_ROOT}')  # Show the active repository root.
print(f'Silver directory exists: {SILVER_DIR.exists()}')  # Confirm Silver inputs are available.
print(f'Rules file exists: {SOURCE_CLEANING_RULES_PATH.exists()}')  # Confirm the cleaning rules are available.
print(f'Cleaned feature directory exists: {CLEAN_DIR.exists()}')  # Confirm Step 03c output folder state.
print(f'Cleaned feature files: {len(list(CLEAN_DIR.glob("*.csv")))}')  # Count cleaned source files.


Project root: /content/HDX-sources-and-more-API-connection
Silver directory exists: True
Rules file exists: True
Cleaned feature directory exists: True
Cleaned feature files: 9


In [4]:
cleaned_paths = list(CLEAN_DIR.glob('*.csv'))  # Find existing Step 03c cleaned outputs.

if not cleaned_paths:  # Fresh Colab runtimes usually do not have generated outputs yet.
    if not SILVER_DIR.exists():  # Stop clearly when source files are missing.
        raise FileNotFoundError(f'Silver input directory not found: {SILVER_DIR}')
    if not SOURCE_CLEANING_RULES_PATH.exists():  # Stop clearly when rules are missing.
        raise FileNotFoundError(f'Cleaning rules file not found: {SOURCE_CLEANING_RULES_PATH}')

    print('No cleaned 03c CSVs found, so this notebook is running Step 03c first.')
    summary_report, reject_rows = clean_silver_directory()  # Generate cleaned country-month outputs.

    errors = summary_report[summary_report['status'].eq('error')]  # Identify source-level cleaning failures.
    bad_keys = summary_report[summary_report['key_unique'].eq(False)]  # Identify country-month uniqueness failures.
    if len(errors) or len(bad_keys):  # Stop when 03c failed validation.
        display(summary_report)
        raise ValueError('03c cleaning validation failed; inspect summary_report before running 03d.')

cleaned_paths = sorted(CLEAN_DIR.glob('*.csv'))  # Refresh the cleaned-output file list.
print(f'Cleaned CSVs ready for 03d: {len(cleaned_paths)}')  # Confirm inputs are ready.
print(cleaned_paths[:5])  # Show a quick path preview.


Cleaned CSVs ready for 03d: 9
[PosixPath('/content/HDX-sources-and-more-API-connection/outputs/clean_country_month_year/clean_civilian_targeting_country_month.csv'), PosixPath('/content/HDX-sources-and-more-API-connection/outputs/clean_country_month_year/clean_demonstration_events_country_month.csv'), PosixPath('/content/HDX-sources-and-more-API-connection/outputs/clean_country_month_year/clean_gdacs_country_month.csv'), PosixPath('/content/HDX-sources-and-more-API-connection/outputs/clean_country_month_year/clean_inform_risk_country_month.csv'), PosixPath('/content/HDX-sources-and-more-API-connection/outputs/clean_country_month_year/clean_political_violence_country_month.csv')]


In [5]:
ensure_output_directories()  # Create generated output folders only.

assembled_features, assembly_report, feature_catalog = assemble_feature_table()  # Assemble all cleaned sources into one wide table.

assembly_report  # Display source-level assembly validation.


Base sources merged: 7
Wide sources deferred: ['clean_whs2026_country_month.csv', 'clean_worldriskindex_country_month.csv']
Base feature rows: 46,522
Base feature columns: 24
Wrote: /content/HDX-sources-and-more-API-connection/outputs/model_features/base_feature_table_country_month_year.csv


,source_file,input_rows,feature_columns,key_unique,duplicate_key_rows,missing_key_columns,assembly_role,included_in_base_table,assembled_output_name,assembled_rows,assembled_columns,assembled_key_unique,assembled_duplicate_key_rows,assembled_missing_key_columns
0,clean_civilian_targeting_country_month.csv,6060.0,2,True,0.0,,base_feature_table,True,base_feature_table_country_month_year.csv,46522,24,True,0,
1,clean_demonstration_events_country_month.csv,6060.0,1,True,0.0,,base_feature_table,True,base_feature_table_country_month_year.csv,46522,24,True,0,
2,clean_gdacs_country_month.csv,58.0,3,True,0.0,,base_feature_table,True,base_feature_table_country_month_year.csv,46522,24,True,0,
3,clean_inform_risk_country_month.csv,22920.0,5,True,0.0,,base_feature_table,True,base_feature_table_country_month_year.csv,46522,24,True,0,
4,clean_political_violence_country_month.csv,6060.0,2,True,0.0,,base_feature_table,True,base_feature_table_country_month_year.csv,46522,24,True,0,
5,clean_views_conflict_forecasts_country_month.csv,1000.0,3,True,0.0,,base_feature_table,True,base_feature_table_country_month_year.csv,46522,24,True,0,
6,clean_who_covid_country_month.csv,18881.0,4,True,0.0,,base_feature_table,True,base_feature_table_country_month_year.csv,46522,24,True,0,
7,clean_whs2026_country_month.csv,NaN,8056,None,NaN,,wide_feature_block_later,False,base_feature_table_country_month_year.csv,46522,24,True,0,
8,clean_worldriskindex_country_month.csv,NaN,246,None,NaN,,wide_feature_block_later,False,base_feature_table_country_month_year.csv,46522,24,True,0,


In [6]:
print(f'Assembled rows: {len(assembled_features)}')  # Print final row count.
print(f'Assembled columns: {len(assembled_features.columns)}')  # Print final column count.
print(f'Feature columns: {len(assembled_features.columns) - 4}')  # Print non-key feature count.

display(assembled_features.head())  # Preview the first assembled rows.
display(feature_catalog.head(20))  # Preview the source-to-feature mapping.


Assembled rows: 46522
Assembled columns: 24
Feature columns: 20


,iso3,country,year,month,civilian_targeting_events,civilian_targeting_fatalities,demonstration_events_events,gdacs_event_count,gdacs_max_severity_value,gdacs_mean_severity_value,...,inform_risk_annual_carried_monthly,political_violence_events,political_violence_fatalities,views_conflict_forecasts_views_main_mean,views_conflict_forecasts_views_main_dich,views_conflict_forecasts_views_main_mean_ln,who_covid_covid_new_cases,who_covid_covid_new_deaths,who_covid_covid_cumulative_cases,who_covid_covid_cumulative_deaths
0,AD,Andorra,2020,1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
1,AD,Andorra,2020,2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
2,AD,Andorra,2020,3,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,376.0,12.0,376.0,12.0
3,AD,Andorra,2020,4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,368.0,30.0,744.0,42.0
4,AD,Andorra,2020,5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,20.0,9.0,764.0,51.0


,source_file,original_column,assembled_column
0,clean_civilian_targeting_country_month.csv,events,civilian_targeting_events
1,clean_civilian_targeting_country_month.csv,fatalities,civilian_targeting_fatalities
2,clean_demonstration_events_country_month.csv,events,demonstration_events_events
3,clean_gdacs_country_month.csv,gdacs_event_count,gdacs_event_count
4,clean_gdacs_country_month.csv,gdacs_max_severity_value,gdacs_max_severity_value
5,clean_gdacs_country_month.csv,gdacs_mean_severity_value,gdacs_mean_severity_value
6,clean_inform_risk_country_month.csv,inform_cc,inform_risk_inform_cc
7,clean_inform_risk_country_month.csv,inform_ha,inform_risk_inform_ha
8,clean_inform_risk_country_month.csv,inform_inform,inform_risk_inform_inform
9,clean_inform_risk_country_month.csv,inform_vu,inform_risk_inform_vu


In [7]:
source_key_errors = assembly_report[assembly_report['key_unique'].eq(False)]  # Identify cleaned source tables with duplicate keys.
assembled_key_errors = assembly_report[assembly_report['assembled_key_unique'].eq(False)]  # Identify final assembled duplicate-key failures.
missing_feature_catalog = feature_catalog[feature_catalog['assembled_column'].isna()] if 'assembled_column' in feature_catalog.columns else feature_catalog  # Guard against catalog schema failures.

print(f'Source tables with duplicate country-month keys: {len(source_key_errors)}')  # Print source key issue count.
print(f'Assembled duplicate-key failures: {len(assembled_key_errors)}')  # Print final key issue count.
print(f'Catalog rows missing assembled column names: {len(missing_feature_catalog)}')  # Print catalog issue count.

if len(source_key_errors) or len(assembled_key_errors) or len(missing_feature_catalog):  # Stop when assembly failed validation.
    raise ValueError('03d feature assembly validation failed; inspect assembly_report and feature_catalog.')  # Raise a visible notebook error.

print('03d smoke test passed: assembled feature table and reports were written.')  # Confirm successful run.


Source tables with duplicate country-month keys: 0
Assembled duplicate-key failures: 0
Catalog rows missing assembled column names: 0
03d smoke test passed: assembled feature table and reports were written.


In [8]:
print(f'Base feature table: {MODEL_FEATURES_DIR / "base_feature_table_country_month_year.csv"}')  # Show base table path.
print(f'Assembly report: {FEATURE_REPORTS_DIR / "feature_assembly_report.csv"}')  # Show assembly report path.
print(f'Feature catalog: {FEATURE_REPORTS_DIR / "feature_column_catalog.csv"}')  # Show feature catalog path.


Base feature table: /content/HDX-sources-and-more-API-connection/outputs/model_features/base_feature_table_country_month_year.csv
Assembly report: /content/HDX-sources-and-more-API-connection/outputs/feature_reports/feature_assembly_report.csv
Feature catalog: /content/HDX-sources-and-more-API-connection/outputs/feature_reports/feature_column_catalog.csv


In [9]:
duplicate_key_rows = assembled_features.duplicated(["iso3", "country", "year", "month"]).sum()  # Count duplicate keys.

print(f"Duplicate country-month rows: {duplicate_key_rows}")  # Show duplicate count.
print(f"Rows: {len(assembled_features):,}")  # Show row count.
print(f"Columns: {len(assembled_features.columns):,}")  # Show column count.
print(assembly_report[["source_file", "assembly_role", "included_in_base_table"]])  # Show source inclusion.

assert duplicate_key_rows == 0  # Confirm no duplicate keys.

Duplicate country-month rows: 0
Rows: 46,522
Columns: 24
                                        source_file             assembly_role  \
0        clean_civilian_targeting_country_month.csv        base_feature_table   
1      clean_demonstration_events_country_month.csv        base_feature_table   
2                     clean_gdacs_country_month.csv        base_feature_table   
3               clean_inform_risk_country_month.csv        base_feature_table   
4        clean_political_violence_country_month.csv        base_feature_table   
5  clean_views_conflict_forecasts_country_month.csv        base_feature_table   
6                 clean_who_covid_country_month.csv        base_feature_table   
7                   clean_whs2026_country_month.csv  wide_feature_block_later   
8            clean_worldriskindex_country_month.csv  wide_feature_block_later   

   included_in_base_table  
0                    True  
1                    True  
2                    True  
3                   

In [10]:
from pathlib import Path  # Use pathlib for file checks.

model_dir = Path("/content/HDX-sources-and-more-API-connection/outputs/model_features")  # Model outputs.
report_dir = Path("/content/HDX-sources-and-more-API-connection/outputs/feature_reports")  # Report outputs.

print(list(model_dir.glob("*.csv")))  # Should show base feature table.
print(list(report_dir.glob("*.csv")))  # Should show assembly report/catalog.

[PosixPath('/content/HDX-sources-and-more-API-connection/outputs/model_features/base_feature_table_country_month_year.csv')]
[PosixPath('/content/HDX-sources-and-more-API-connection/outputs/feature_reports/feature_assembly_report.csv'), PosixPath('/content/HDX-sources-and-more-API-connection/outputs/feature_reports/feature_column_catalog.csv')]
